# Version 3: Cleaned Retrieval Corpus and Page-Aware Evaluation

Version 3 improves the baseline RAG retrieval system by addressing two issues observed in Version 2:

1. Noisy chunks such as Table of Contents pages and repeated report headers can cause false positive retrieval results.
2. Keyword-only evaluation overestimates retrieval quality because a chunk may contain the query keyword without actually answering the question.

To address these issues, this version:
- cleans noisy chunks from the retrieval corpus,
- rebuilds embeddings and FAISS cosine index on the cleaned corpus,
- applies cross-encoder reranking,
- evaluates retrieval using both manually labeled relevant page ranges and expected evidence keywords,
- reports Recall@K, Precision@K, MRR@K, and per-query error analysis.

## Configuration

In [8]:
import fitz
import pandas as pd
import numpy as np
import faiss
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

pdf_path = "Artificial_ Intelligence_Index_Report_2025.pdf"

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_idx, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({
            "page": page_idx + 1,
            "text": text
        })

    return pages

pages = extract_pdf_text(pdf_path)

print("Number of pages:", len(pages))
print(pages[0]["text"][:1000])

page_df = pd.DataFrame(pages)
page_df["text_length"] = page_df["text"].apply(len)

page_df[["page", "text_length"]].head(20)
page_df["text_length"].describe()

Number of pages: 457
Artificial Intelligence
Index Report 2025



count     457.000000
mean     1797.470460
std       932.849212
min        42.000000
25%      1092.000000
50%      1663.000000
75%      2316.000000
max      4690.000000
Name: text_length, dtype: float64

In [10]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if len(chunk) > 100:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [12]:
all_chunks = []

for page in pages:
    chunks = chunk_text(
        page["text"],
        chunk_size=800,
        overlap=150
    )

    for chunk_id, chunk in enumerate(chunks):
        all_chunks.append({
            "source": "Stanford AI Index Report 2025",
            "page": page["page"],
            "chunk_id": chunk_id,
            "text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_df))
chunks_df.head()

Total chunks: 1421


,source,page,chunk_id,text
0,Stanford AI Index Report 2025,2,0,Artificial Intelligence\nIndex Report 2025\n1\...
1,Stanford AI Index Report 2025,2,1,n offshoot of the One Hundred Year Study of Ar...
2,Stanford AI Index Report 2025,2,2,the rapid evolution of underlying technologies...
3,Stanford AI Index Report 2025,2,3,the world. We have briefed companies like Acce...
4,Stanford AI Index Report 2025,3,0,Artificial Intelligence\nIndex Report 2025\n2\...


In [16]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [18]:
print("chunks_df shape:", chunks_df.shape)
print(chunks_df.columns)
print(chunks_df.head())

chunks_df shape: (1421, 4)
Index(['source', 'page', 'chunk_id', 'text'], dtype='object')
                          source  page  chunk_id  \
0  Stanford AI Index Report 2025     2         0   
1  Stanford AI Index Report 2025     2         1   
2  Stanford AI Index Report 2025     2         2   
3  Stanford AI Index Report 2025     2         3   
4  Stanford AI Index Report 2025     3         0   

                                                text  
0  Artificial Intelligence\nIndex Report 2025\n1\...  
1  n offshoot of the One Hundred Year Study of Ar...  
2  the rapid evolution of underlying technologies...  
3  the world. We have briefed companies like Acce...  
4  Artificial Intelligence\nIndex Report 2025\n2\...  


In [20]:
print(embedding_model)
print(reranker)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'BertForSequenceClassification'})
)


## Clean Noisy Chunks

In [25]:
def clean_chunk_text(text):
    """
    Clean repeated headers, page numbers, excessive whitespace, and common PDF artifacts.
    """
    if pd.isna(text):
        return ""

    lines = str(text).split("\n")
    cleaned_lines = []

    noise_patterns = [
        r"^artificial intelligence index report 2025$",
        r"^artificial intelligence index report$",
        r"^artificial intelligence$",
        r"^index report 2025$",
        r"^stanford university$",
        r"^hai$",
        r"^chapter \d+$",
        r"^\d+$"
    ]

    for line in lines:
        line = line.strip()

        if not line:
            continue

        line_lower = line.lower()

        is_noise = False
        for pattern in noise_patterns:
            if re.fullmatch(pattern, line_lower):
                is_noise = True
                break

        if is_noise:
            continue

        cleaned_lines.append(line)

    cleaned_text = " ".join(cleaned_lines)

    # Normalize whitespace
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    return cleaned_text

## Detect and Remove Noisy Chunks

In [28]:
def is_noisy_chunk(text):
    """
    Identify chunks that are likely not useful for retrieval.
    """
    if pd.isna(text):
        return True

    text = str(text).strip()
    text_lower = text.lower()

    # Too short to be meaningful
    if len(text) < 200:
        return True

    # Table of contents usually causes false positives
    toc_patterns = [
        "table of contents",
        "contents chapter",
        "chapter 1",
        "chapter 2",
        "chapter 3",
        "chapter 4",
        "chapter 5",
        "chapter 6",
        "chapter 7",
        "chapter 8",
        "chapter 9",
        "chapter 10"
    ]

    if "table of contents" in text_lower:
        return True

    # Chunks dominated by dot leaders or section listings
    if text_lower.count("chapter") >= 5 and len(text_lower) < 1200:
        return True

    # Chunks with too many page-number-like tokens may be TOC/index
    digit_tokens = re.findall(r"\b\d{1,4}\b", text_lower)
    words = re.findall(r"[a-zA-Z]+", text_lower)

    if len(words) > 0:
        digit_ratio = len(digit_tokens) / len(words)
        if digit_ratio > 0.35:
            return True

    return False

In [30]:
chunks_df_clean = chunks_df.copy()

chunks_df_clean["clean_text"] = chunks_df_clean["text"].apply(clean_chunk_text)
chunks_df_clean["is_noisy"] = chunks_df_clean["clean_text"].apply(is_noisy_chunk)

print("Original chunks:", len(chunks_df_clean))
print("Noisy chunks:", chunks_df_clean["is_noisy"].sum())

chunks_df_clean = chunks_df_clean[~chunks_df_clean["is_noisy"]].copy()
chunks_df_clean = chunks_df_clean.reset_index(drop=True)

print("Clean chunks:", len(chunks_df_clean))

Original chunks: 1421
Noisy chunks: 638
Clean chunks: 783


In [32]:
noisy_examples = chunks_df.copy()
noisy_examples["clean_text"] = noisy_examples["text"].apply(clean_chunk_text)
noisy_examples["is_noisy"] = noisy_examples["clean_text"].apply(is_noisy_chunk)

noisy_examples[noisy_examples["is_noisy"]][["page", "chunk_id", "clean_text"]].head(10)

,page,chunk_id,clean_text
19,6,1,"correct solutions exist, limiting their effect..."
33,12,0,Report Highlights Research and Development Tec...
53,17,4,"em, with many expecting misinformation campaig..."
70,21,3,",557 in 2023. Since 2016, the total number of ..."
82,26,0,Table of Contents Overview Chapter Highlights ...
83,26,1,Energy Efficiency and Environmental Impact 1.5...
84,27,0,Table of Contents Chapter 1 Preview This chapt...
85,28,0,Table of Contents Chapter 1 Preview Chapter Hi...
88,28,3,g annually. Large-scale industry investment co...
89,29,0,Table of Contents Chapter 1 Preview Chapter Hi...


In [34]:
chunks_df_clean[["page", "chunk_id", "clean_text"]].head()

,page,chunk_id,clean_text
0,2,0,Welcome to the eighth edition of the AI Index ...
1,2,1,n offshoot of the One Hundred Year Study of Ar...
2,2,2,the rapid evolution of underlying technologies...
3,2,3,the world. We have briefed companies like Acce...
4,3,0,"As AI continues to reshape our lives, the corp..."


## Rebuild Clean Embeddings

In [37]:
clean_texts = chunks_df_clean["clean_text"].tolist()

clean_embeddings = embedding_model.encode(
    clean_texts,
    show_progress_bar=False,
    convert_to_numpy=True
)

clean_embeddings = clean_embeddings.astype("float32")

print("Clean embedding shape:", clean_embeddings.shape)

Clean embedding shape: (783, 384)


## Rebuild FAISS Cosine Index

In [40]:
clean_embeddings_cosine = clean_embeddings.copy()

faiss.normalize_L2(clean_embeddings_cosine)

dimension = clean_embeddings_cosine.shape[1]

index_clean_cosine = faiss.IndexFlatIP(dimension)
index_clean_cosine.add(clean_embeddings_cosine)

print("Number of clean vectors in FAISS index:", index_clean_cosine.ntotal)

Number of clean vectors in FAISS index: 783


## Clean Dense Retrieval Function

In [43]:
def retrieve_dense_cosine_clean(query, top_k=10):
    """
    Retrieve top-k chunks from the cleaned corpus using cosine similarity.
    """
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index_clean_cosine.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0]):
        row = chunks_df_clean.iloc[idx]

        results.append({
            "rank": rank + 1,
            "page": int(row["page"]),
            "chunk_id": int(row["chunk_id"]),
            "cosine_score": float(scores[0][rank]),
            "text": row["clean_text"]
        })

    return results

In [45]:
query = "What does the report say about the cost of AI inference?"

results = retrieve_dense_cosine_clean(query, top_k=5)

for r in results:
    print("=" * 120)
    print(f"Rank: {r['rank']} | Page: {r['page']} | Cosine: {r['cosine_score']:.4f}")
    print("-" * 120)
    print(r["text"][:1200])

Rank: 1 | Page: 2 | Cosine: 0.6258
------------------------------------------------------------------------------------------------------------------------
Welcome to the eighth edition of the AI Index report. The 2025 Index is our most comprehensive to date and arrives at an important moment, as AI’s influence across society, the economy, and global governance continues to intensify. New in this year’s report are in-depth analyses of the evolving landscape of AI hardware, novel estimates of inference costs, and new analyses of AI publication and patenting trends. We also introduce fresh data on corporate adoption of responsible AI practices, along with expanded coverage of AI’s growing role in science and medicine. Since its founding in 2017 as an offshoot of the One Hundred Year Study of Artificial Intelligence, the AI Index has been committed to equipping policymakers, journalists, executiv
Rank: 2 | Page: 65 | Cosine: 0.5775
---------------------------------------------------------

## Clean Retrieval + Cross-Encoder Reranking

In [50]:
def retrieve_with_rerank_clean(query, first_k=20, top_k=5):
    """
    First retrieve candidate chunks using clean dense retrieval,
    then rerank them with a cross-encoder.
    """
    candidates = retrieve_dense_cosine_clean(query, top_k=first_k)

    pairs = [(query, item["text"]) for item in candidates]

    rerank_scores = reranker.predict(pairs)

    for item, score in zip(candidates, rerank_scores):
        item["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    for i, item in enumerate(reranked[:top_k]):
        item["rank"] = i + 1

    return reranked[:top_k]

In [52]:
query = "What does the report say about the cost of AI inference?"

reranked_results = retrieve_with_rerank_clean(
    query,
    first_k=20,
    top_k=5
)

for r in reranked_results:
    print("=" * 120)
    print(
        f"Rank: {r['rank']} | Page: {r['page']} | "
        f"Cosine: {r['cosine_score']:.4f} | Rerank: {r['rerank_score']:.4f}"
    )
    print("-" * 120)
    print(r["text"][:1200])

Rank: 1 | Page: 65 | Cosine: 0.5775 | Rerank: 4.5585
------------------------------------------------------------------------------------------------------------------------
token prices. To analyze inference costs, the AI Index worked with Epoch to measure how costs have decreased for a fixed AI performance threshold. This standardized approach facilitates a more accurate comparison. While newer models may cost more, they also tend to perform significantly better—so comparing them directly to older, less capable models can obscure the real trend: AI performance per dollar has improved substantially. For instance, the inference cost for an AI model scoring the equivalent of GPT-3.5 (64.8) on MMLU, a popular benchmark for assessing language model performance, dropped from $20 per million tokens in November 2022 to just $0.07 per million tokens by October 2024 (Gemini-1.5-Flash-8B)—a more than 280-fold reduction in approximately 1.5 years. A similar
Rank: 2 | Page: 13 | Cosine: 0.5646 | 

## Clean Display Function

In [55]:
def show_clean_reranked_results(query, first_k=20, top_k=5):
    results = retrieve_with_rerank_clean(
        query=query,
        first_k=first_k,
        top_k=top_k
    )

    display_df = pd.DataFrame([
        {
            "rank": r["rank"],
            "page": r["page"],
            "chunk_id": r["chunk_id"],
            "cosine_score": round(r["cosine_score"], 4),
            "rerank_score": round(r["rerank_score"], 4),
            "preview": r["text"][:350].replace("\n", " ")
        }
        for r in results
    ])

    return display_df

In [57]:
show_clean_reranked_results(
    "How has AI investment changed in recent years?",
    first_k=20,
    top_k=5
)

,rank,page,chunk_id,cosine_score,rerank_score,preview
0,1,260,2,0.6621,3.8313,.3.16 presents trends over time in AI focus ar...
1,2,3,1,0.7192,3.4426,"anwhile, AI adoption has accelerated at an unp..."
2,3,18,2,0.7000,2.9354,ing more than thirteenfold since 2014. 2. Gene...
3,4,18,1,0.7008,2.6069,academic researchers. The number of RAI papers...
4,5,364,3,0.7040,2.4113,"teenfold, from $230 million to $4.5 billion. F..."


## Build Improved Evaluation Benchmark

The expected_pages need to be manually checked based on the actual content of the PDF. The version below is an initial version based on current search results and common sections of the AI Index, and can be further revised later.

In [86]:
eval_questions_v3 = [
    {
        "question": "How has AI investment changed in recent years?",
        "expected_pages": [3, 18, 260, 364],
        "expected_keywords": ["investment", "private investment"],
        "question_type": "trend"
    },
    {
        "question": "What does the report say about the cost of AI inference?",
        "expected_pages": [2, 5, 13, 65, 100],
        "expected_keywords": ["inference", "cost"],
        "question_type": "trend"
    },
    {
        "question": "What are the trends in AI publications and patents?",
        "expected_pages": [2, 13, 14, 29, 30],
        "expected_keywords": ["publications", "patents"],
        "question_type": "trend"
    },
    {
        "question": "How is AI being used in healthcare?",
        "expected_pages": [167, 298, 310, 318],
        "expected_keywords": ["healthcare", "medical"],
        "question_type": "application"
    },
    {
        "question": "What risks or challenges are associated with AI adoption?",
        "expected_pages": [17, 180, 182, 183, 185],
        "expected_keywords": ["risk", "adoption"],
        "question_type": "risk"
    },
    {
        "question": "What does the report say about foundation models?",
        "expected_pages": [165, 201, 297, 443],
        "expected_keywords": ["foundation model", "model"],
        "question_type": "concept"
    },
    {
        "question": "How are AI models evaluated in the report?",
        "expected_pages": [47, 102, 103, 201],
        "expected_keywords": ["benchmark", "evaluation"],
        "question_type": "method"
    },
    {
        "question": "What does the report say about AI education?",
        "expected_pages": [378, 379, 382, 394],
        "expected_keywords": ["education", "AI"],
        "question_type": "application"
    },
    {
        "question": "What are the trends in responsible AI?",
        "expected_pages": [2, 4, 16, 18, 167],
        "expected_keywords": ["responsible AI", "risk"],
        "question_type": "risk"
    },
    {
        "question": "What does the report say about AI regulation?",
        "expected_pages": [351, 399, 412, 413],
        "expected_keywords": ["regulation", "policy"],
        "question_type": "policy"
    }
]

eval_df_v3 = pd.DataFrame(eval_questions_v3)

eval_df_v3

,question,expected_pages,expected_keywords,question_type
0,How has AI investment changed in recent years?,"[3, 18, 260, 364]","[investment, private investment]",trend
1,What does the report say about the cost of AI ...,"[2, 5, 13, 65, 100]","[inference, cost]",trend
2,What are the trends in AI publications and pat...,"[2, 13, 14, 29, 30]","[publications, patents]",trend
3,How is AI being used in healthcare?,"[167, 298, 310, 318]","[healthcare, medical]",application
4,What risks or challenges are associated with A...,"[17, 180, 182, 183, 185]","[risk, adoption]",risk
5,What does the report say about foundation models?,"[165, 201, 297, 443]","[foundation model, model]",concept
6,How are AI models evaluated in the report?,"[47, 102, 103, 201]","[benchmark, evaluation]",method
7,What does the report say about AI education?,"[378, 379, 382, 394]","[education, AI]",application
8,What are the trends in responsible AI?,"[2, 4, 16, 18, 167]","[responsible AI, risk]",risk
9,What does the report say about AI regulation?,"[351, 399, 412, 413]","[regulation, policy]",policy


## Helper: Inspect Candidate Pages Before Finalizing Benchmark

In [64]:
def inspect_query_results(query, top_k=10, use_rerank=True):
    if use_rerank:
        results = retrieve_with_rerank_clean(query, first_k=30, top_k=top_k)
    else:
        results = retrieve_dense_cosine_clean(query, top_k=top_k)

    for r in results:
        print("=" * 120)
        if use_rerank:
            print(
                f"Rank: {r['rank']} | Page: {r['page']} | "
                f"Cosine: {r['cosine_score']:.4f} | Rerank: {r['rerank_score']:.4f}"
            )
        else:
            print(
                f"Rank: {r['rank']} | Page: {r['page']} | "
                f"Cosine: {r['cosine_score']:.4f}"
            )
        print("-" * 120)
        print(r["text"][:1500])
        print("\n")

In [84]:
inspect_query_results(
    "What does the report say about AI regulation?",
    top_k=5,
    use_rerank=True
)

Rank: 1 | Page: 412 | Cosine: 0.6344 | Rerank: 2.9998
------------------------------------------------------------------------------------------------------------------------
n 2022 and 2023, the study gathered responses from approximately 1,000 policymakers. Its timing allowed researchers to compare how policymakers’ views on AI shifted before and after the launch of ChatGPT. Figure 8.2.1 illustrates the extent to which local policymakers agree with the statement: AI should be regulated by the government. In 2023, 73.7% of local U.S. policymakers supported this view, a significant increase from 55.7% in 2022. The launch of ChatGPT appears to have played a key role in shifting policymaker sentiment toward regulation. Support for AI regulation was higher among Democrats (79.2%) than Republicans (55.5%), though both groups registered a notable increase after 2022. 8.2 US Policymaker Opinion Chapter 8: Public Opinion Figure 8.2.1 64.50% 73.70% 55.70%


Rank: 2 | Page: 413 | Cosine: 0.6318

## Page + Keyword Relevance Functions

In [89]:
def page_hit(result_page, expected_pages, tolerance=1):
    """
    A page is considered correct if it is within tolerance of any expected page.
    """
    for expected_page in expected_pages:
        if abs(result_page - expected_page) <= tolerance:
            return True
    return False


def keyword_hit_min_matches(text, keywords, min_matches=1):
    """
    Check whether the retrieved text contains at least min_matches expected keywords.
    """
    text_lower = text.lower()
    matched = []

    for keyword in keywords:
        if keyword.lower() in text_lower:
            matched.append(keyword)

    return len(matched) >= min_matches, matched


def is_relevant_result(
    result,
    expected_pages,
    expected_keywords,
    tolerance=1,
    min_keyword_matches=1
):
    """
    A result is relevant only if both page range and keyword evidence match.
    """
    page_ok = page_hit(
        result_page=result["page"],
        expected_pages=expected_pages,
        tolerance=tolerance
    )

    keyword_ok, matched_keywords = keyword_hit_min_matches(
        text=result["text"],
        keywords=expected_keywords,
        min_matches=min_keyword_matches
    )

    is_relevant = page_ok and keyword_ok

    return is_relevant, matched_keywords

## Evaluation Metrics: Recall@K, Precision@K, MRR@K

In [92]:
def evaluate_recall_at_k_page_keyword(
    eval_df,
    retrieve_func,
    k=5,
    tolerance=1,
    min_keyword_matches=1
):
    hits = 0
    rows = []

    for _, row in eval_df.iterrows():
        query = row["question"]
        expected_pages = row["expected_pages"]
        expected_keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)

        relevant_found = False
        hit_pages = []
        matched_all = []

        for r in results:
            relevant, matched = is_relevant_result(
                r,
                expected_pages=expected_pages,
                expected_keywords=expected_keywords,
                tolerance=tolerance,
                min_keyword_matches=min_keyword_matches
            )

            if relevant:
                relevant_found = True
                hit_pages.append(r["page"])
                matched_all.extend(matched)

        if relevant_found:
            hits += 1

        rows.append({
            "question": query,
            "recall_hit": int(relevant_found),
            "hit_pages": hit_pages,
            "matched_keywords": list(set(matched_all))
        })

    recall = hits / len(eval_df)

    return recall, pd.DataFrame(rows)

In [94]:
def evaluate_precision_at_k_page_keyword(
    eval_df,
    retrieve_func,
    k=5,
    tolerance=1,
    min_keyword_matches=1
):
    precision_scores = []
    rows = []

    for _, row in eval_df.iterrows():
        query = row["question"]
        expected_pages = row["expected_pages"]
        expected_keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)

        relevant_count = 0
        relevant_pages = []

        for r in results:
            relevant, matched = is_relevant_result(
                r,
                expected_pages=expected_pages,
                expected_keywords=expected_keywords,
                tolerance=tolerance,
                min_keyword_matches=min_keyword_matches
            )

            if relevant:
                relevant_count += 1
                relevant_pages.append(r["page"])

        precision = relevant_count / k
        precision_scores.append(precision)

        rows.append({
            "question": query,
            "precision_at_k": precision,
            "relevant_count": relevant_count,
            "relevant_pages": relevant_pages
        })

    mean_precision = sum(precision_scores) / len(precision_scores)

    return mean_precision, pd.DataFrame(rows)

In [96]:
def evaluate_mrr_at_k_page_keyword(
    eval_df,
    retrieve_func,
    k=5,
    tolerance=1,
    min_keyword_matches=1
):
    reciprocal_ranks = []
    rows = []

    for _, row in eval_df.iterrows():
        query = row["question"]
        expected_pages = row["expected_pages"]
        expected_keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)

        first_hit_rank = None
        hit_page = None
        matched_keywords = []
        hit_preview = None

        for i, r in enumerate(results):
            relevant, matched = is_relevant_result(
                r,
                expected_pages=expected_pages,
                expected_keywords=expected_keywords,
                tolerance=tolerance,
                min_keyword_matches=min_keyword_matches
            )

            if relevant:
                first_hit_rank = i + 1
                hit_page = r["page"]
                matched_keywords = matched
                hit_preview = r["text"][:300].replace("\n", " ")
                break

        reciprocal_rank = 0 if first_hit_rank is None else 1 / first_hit_rank
        reciprocal_ranks.append(reciprocal_rank)

        rows.append({
            "question": query,
            "first_hit_rank": first_hit_rank,
            "reciprocal_rank": reciprocal_rank,
            "hit_page": hit_page,
            "matched_keywords": matched_keywords,
            "hit_preview": hit_preview
        })

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return mrr, pd.DataFrame(rows)

## Create Retrieval Wrappers

In [99]:
def retrieve_clean_dense_wrapper(query, top_k=5):
    return retrieve_dense_cosine_clean(query, top_k=top_k)


def retrieve_clean_rerank_wrapper(query, top_k=5):
    return retrieve_with_rerank_clean(query, first_k=20, top_k=top_k)

## Run Evaluation

In [102]:
K = 5
TOLERANCE = 1
MIN_KEYWORD_MATCHES = 1

#### Dense Retrieval Evaluation

In [106]:
dense_recall, dense_recall_detail = evaluate_recall_at_k_page_keyword(
    eval_df_v3,
    retrieve_func=retrieve_clean_dense_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

dense_precision, dense_precision_detail = evaluate_precision_at_k_page_keyword(
    eval_df_v3,
    retrieve_func=retrieve_clean_dense_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

dense_mrr, dense_mrr_detail = evaluate_mrr_at_k_page_keyword(
    eval_df_v3,
    retrieve_func=retrieve_clean_dense_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

print("Clean Dense Recall@5:", dense_recall)
print("Clean Dense Precision@5:", dense_precision)
print("Clean Dense MRR@5:", dense_mrr)

Clean Dense Recall@5: 1.0
Clean Dense Precision@5: 0.6
Clean Dense MRR@5: 0.8666666666666666


#### Reranked Retrieval Evaluation

In [109]:
rerank_recall, rerank_recall_detail = evaluate_recall_at_k_page_keyword(
    eval_df_v3,
    retrieve_func=retrieve_clean_rerank_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

rerank_precision, rerank_precision_detail = evaluate_precision_at_k_page_keyword(
    eval_df_v3,
    retrieve_func=retrieve_clean_rerank_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

rerank_mrr, rerank_mrr_detail = evaluate_mrr_at_k_page_keyword(
    eval_df_v3,
    retrieve_func=retrieve_clean_rerank_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

print("Clean Reranked Recall@5:", rerank_recall)
print("Clean Reranked Precision@5:", rerank_precision)
print("Clean Reranked MRR@5:", rerank_mrr)

Clean Reranked Recall@5: 1.0
Clean Reranked Precision@5: 0.86
Clean Reranked MRR@5: 0.95


## Build Comparison Table

In [115]:
summary_results = pd.DataFrame([
    {
        "method": "Clean Dense Retrieval",
        "Recall@5": dense_recall,
        "Precision@5": dense_precision,
        "MRR@5": dense_mrr
    },
    {
        "method": "Clean Dense Retrieval + Cross-Encoder Reranking",
        "Recall@5": rerank_recall,
        "Precision@5": rerank_precision,
        "MRR@5": rerank_mrr
    }
])

summary_results

,method,Recall@5,Precision@5,MRR@5
0,Clean Dense Retrieval,1.0,0.60,0.866667
1,Clean Dense Retrieval + Cross-Encoder Reranking,1.0,0.86,0.950000


## Per-Query Error Analysis

In [118]:
dense_detail_for_compare = dense_mrr_detail[
    ["question", "first_hit_rank", "reciprocal_rank", "hit_page", "matched_keywords", "hit_preview"]
].copy()

rerank_detail_for_compare = rerank_mrr_detail[
    ["question", "first_hit_rank", "reciprocal_rank", "hit_page", "matched_keywords", "hit_preview"]
].copy()

comparison_v3 = dense_detail_for_compare.merge(
    rerank_detail_for_compare,
    on="question",
    suffixes=("_dense", "_rerank")
)

comparison_v3["mrr_change"] = (
    comparison_v3["reciprocal_rank_rerank"] -
    comparison_v3["reciprocal_rank_dense"]
)

comparison_v3

,question,first_hit_rank_dense,reciprocal_rank_dense,hit_page_dense,matched_keywords_dense,hit_preview_dense,first_hit_rank_rerank,reciprocal_rank_rerank,hit_page_rerank,matched_keywords_rerank,hit_preview_rerank,mrr_change
0,How has AI investment changed in recent years?,1,1.000000,3,[investment],"anwhile, AI adoption has accelerated at an unp...",1,1.0,260,[investment],.3.16 presents trends over time in AI focus ar...,0.000000
1,What does the report say about the cost of AI ...,1,1.000000,2,"[inference, cost]",Welcome to the eighth edition of the AI Index ...,1,1.0,65,"[inference, cost]","token prices. To analyze inference costs, the ...",0.000000
2,What are the trends in AI publications and pat...,1,1.000000,28,[publications],"otals, while the United States leads in highly...",1,1.0,30,[publications],"larly high-impact research. This year, the AI ...",0.000000
3,How is AI being used in healthcare?,3,0.333333,310,[medical],"lly over the past decade, highlighted by the d...",2,0.5,310,[medical],"lly over the past decade, highlighted by the d...",0.166667
4,What risks or challenges are associated with A...,1,1.000000,183,[risk],"iversity and nondiscrimination risks (e.g., fa...",1,1.0,185,"[risk, adoption]",adoption” “Closed-source models are safer”/ “O...,0.000000
5,What does the report say about foundation models?,1,1.000000,201,"[foundation model, model]","system; and downstream, encompassing applicati...",1,1.0,201,"[foundation model, model]","system; and downstream, encompassing applicati...",0.000000
6,How are AI models evaluated in the report?,3,0.333333,102,"[benchmark, evaluation]",may be coming to an end. To truly assess the c...,1,1.0,102,"[benchmark, evaluation]",may be coming to an end. To truly assess the c...,0.666667
7,What does the report say about AI education?,1,1.000000,394,"[education, AI]","ntelligence and Education, it offered specific...",1,1.0,378,"[education, AI]",. The U.S. Department of Education’s Office of...,0.000000
8,What are the trends in responsible AI?,1,1.000000,16,[responsible AI],"kernels, while delivering results faster and a...",1,1.0,167,[responsible AI],"tions In this chapter, the AI Index explores f...",0.000000
9,What does the report say about AI regulation?,1,1.000000,351,"[regulation, policy]","scope, regulatory intent, and originating agen...",1,1.0,412,"[regulation, policy]","n 2022 and 2023, the study gathered responses ...",0.000000


In [120]:
comparison_v3[comparison_v3["mrr_change"] > 0]

,question,first_hit_rank_dense,reciprocal_rank_dense,hit_page_dense,matched_keywords_dense,hit_preview_dense,first_hit_rank_rerank,reciprocal_rank_rerank,hit_page_rerank,matched_keywords_rerank,hit_preview_rerank,mrr_change
3,How is AI being used in healthcare?,3,0.333333,310,[medical],"lly over the past decade, highlighted by the d...",2,0.5,310,[medical],"lly over the past decade, highlighted by the d...",0.166667
6,How are AI models evaluated in the report?,3,0.333333,102,"[benchmark, evaluation]",may be coming to an end. To truly assess the c...,1,1.0,102,"[benchmark, evaluation]",may be coming to an end. To truly assess the c...,0.666667


In [122]:
comparison_v3[comparison_v3["mrr_change"] < 0]

,question,first_hit_rank_dense,reciprocal_rank_dense,hit_page_dense,matched_keywords_dense,hit_preview_dense,first_hit_rank_rerank,reciprocal_rank_rerank,hit_page_rerank,matched_keywords_rerank,hit_preview_rerank,mrr_change


In [124]:
comparison_v3[
    comparison_v3["first_hit_rank_rerank"].isna() |
    comparison_v3["first_hit_rank_dense"].isna()
]

,question,first_hit_rank_dense,reciprocal_rank_dense,hit_page_dense,matched_keywords_dense,hit_preview_dense,first_hit_rank_rerank,reciprocal_rank_rerank,hit_page_rerank,matched_keywords_rerank,hit_preview_rerank,mrr_change


In [128]:
chunks_df_clean.to_csv("chunks_clean_v3.csv", index=False)
np.save("clean_embeddings_v3.npy", clean_embeddings)

summary_results.to_csv("evaluation_summary_v3.csv", index=False)
comparison_v3.to_csv("evaluation_comparison_v3.csv", index=False)

print("Version 3 artifacts saved.")

Version 3 artifacts saved.
